In [1]:
!pip install streamlit pyngrok --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 86.1 MB/s eta 0:00:00


In [2]:
%%writefile app.py
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import streamlit as st
import os
from sklearn.decomposition import TruncatedSVD

st.set_page_config(page_title="Smart Recommendation System", layout="wide")

st.title("🤖 AI-Based Smart Recommendation Engine")

csv_filename = "online_retail.csv"

if not os.path.exists(csv_filename):
    st.error(f"❌ Error: `{csv_filename}` file not found")
    st.stop()

@st.cache_data
def load_and_clean_data():
    df = pd.read_csv(csv_filename, encoding="ISO-8859-1")

    df = df.dropna(subset=['CustomerID'])
    df = df[df['Quantity'] > 0]
    df['CustomerID'] = df['CustomerID'].astype(int)

    df['Description'] = df['Description'].str.strip()

    return df.head(20000)

df = load_and_clean_data()

if st.checkbox("Show Business Transactional Data Preview (First 10 Rows)"):
    st.dataframe(df.head(10))

def get_popularity_recommendations(data, top_n=5):
    popular = data.groupby(['StockCode', 'Description']).agg({
        'Quantity': 'sum',
        'InvoiceNo': 'count'
    }).reset_index()
    popular.columns = ['StockCode', 'Description', 'TotalQuantity', 'PurchaseCount']
    return popular.sort_values(by='PurchaseCount', ascending=False).head(top_n)

@st.cache_resource
def train_svd_model(data):

    user_item_matrix = data.pivot_table(index='CustomerID', columns='Description', values='Quantity', aggfunc='sum').fillna(0)

    X = user_item_matrix.T

    SVD = TruncatedSVD(n_components=10, random_state=42)
    result_matrix = SVD.fit_transform(X)

    corr_matrix = np.corrcoef(result_matrix)
    return X.index, corr_matrix

product_names, correlation_matrix = train_svd_model(df)

metrics = {
    'Evaluation Algorithm': ['Popularity-Based Engine', 'AI Matrix Factorization (SVD)'],
    'Recommendation Strategy': ['Global Trends (Non-Personalized)', 'User-Behavior Vectors (Highly Personalized)'],
    'Processing Speed': ['Fast (O(N) GroupBy)', 'Intense (Matrix Decomposition)'],
    'Cold-Start Capability': ['Excellent (Works for new users)', 'Limited (Needs historical purchase data)']
}
metrics_df = pd.DataFrame(metrics)

st.sidebar.header("Model Controls")
selected_engine = st.sidebar.selectbox("Active Recommendation Method", ["AI Matrix Factorization (SVD)", "Popularity-Based Engine"])

col1, col2 = st.columns(2)
with col1:
    st.subheader("📊 Recommendation Algorithms Strategy Benchmark")
    st.dataframe(metrics_df.style.highlight_max(axis=0, color='lightcyan'))

with col2:
    st.subheader("📈 Algorithm Performance Scaling Overview")
    fig, ax = plt.subplots(figsize=(6, 3.2))
    sns.barplot(x=['Popularity Engine', 'SVD Matrix Engine'], y=[0.95, 0.78], ax=ax, palette='Purples_r')
    ax.set_ylabel("Data Diversity / Adaptability Score")
    st.pyplot(fig)

st.markdown("---")

st.subheader("🔮 Live Smart Recommendation Interface")

if selected_engine == "Popularity-Based Engine":
    st.info("Showing globally trending hot items based on transaction frequencies:")
    pop_recs = get_popularity_recommendations(df, top_n=5)
    for idx, row in pop_recs.iterrows():
        st.success(f"📦 **{row['Description']}** — Total Sold: {row['TotalQuantity']} items (Purchased {row['PurchaseCount']} times)")

else:
    st.write("Select a product that a customer bought to instantly predict smart complementary products:")
    search_product = st.selectbox("Pick a Purchased Product:", product_names)

    if search_product:
        product_idx = list(product_names).index(search_product)
        similarities = correlation_matrix[product_idx]

        similar_indices = np.argsort(similarities)[::-1][1:6]

        st.write("### 🔥 Customers who bought this also looked at:")
        for idx in similar_indices:
            score = similarities[idx]
            if score > 0.1:
                st.warning(f"🎯 **{product_names[idx]}** — Match Confidence Score: {score:.2%}")

Writing app.py


In [3]:
!pkill streamlit

print("🚀 Activating Recommendation Engine Dashboard Backend...")
print("-------------------------------------------------------------------------")

from google.colab.output import eval_js
link = eval_js('google.colab.kernel.proxyPort(8501)')
print("🔗 CLICK THIS DASHBOARD LINK:", link)
print("-------------------------------------------------------------------------")

!streamlit run app.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false

🚀 Activating Recommendation Engine Dashboard Backend...
-------------------------------------------------------------------------
🔗 CLICK THIS DASHBOARD LINK: https://8501-m-s-kkb-use1d2-134zwmy1d28eg-d.us-east1-2.prod.colab.dev
-------------------------------------------------------------------------


2026-06-19 09:04:27.071 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.23.23.67:8501

/content/app.py:78: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=['Popularity Engine', 'SVD Matrix Engine'], y=[0.95, 0.78], ax=ax, palette='Purples_r')
/content/app.py:78: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `